# k-means on the STL-10 embeddings

The ViT embeddings of 1,000 labelled STL-10 images: 768 numbers per image, six depths.
Here we take **layer 12** and cluster it into **10** groups which is the same number as the real
classes, but k-means never sees the labels.

The labels are used only afterwards, to check what came out.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import normalized_mutual_info_score, silhouette_score

DATA = "../week_2/data"
IMG = 96

d = np.load(f"{DATA}/embeddings_stl10_1000.npz")

X = d["E_l12"]
y = d["y"]

class_names = [l.strip() for l in open(f"{DATA}/stl10_binary/class_names.txt") if l.strip()]

def load_rgb(split):
    A = np.fromfile(f"{DATA}/stl10_binary/{split}_X.bin", dtype=np.uint8).reshape(-1, 3, IMG, IMG)
    return A.transpose(0, 3, 2, 1)

X_img = np.concatenate([load_rgb("train"), load_rgb("test")])[d["idx"]]

K = 10
RANDOM_STATE = 0

print(X.shape, X_img.shape, y.shape)

### k-means with K = 10

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X)
labels = km.labels_

print("cluster sizes:", np.bincount(labels, minlength=K))

In [ ]:
def report(name, labels):
    print(f"{name}")
    print(f"  NMI        {normalized_mutual_info_score(y, labels):.3f}")
    print(f"  silhouette {silhouette_score(X, labels):.3f}")

### What ended up in each cluster?

Rows are the true classes, columns the clusters k-means found. A perfect result would have
exactly one non-zero entry per row.

In [ ]:
pd.crosstab(pd.Series([class_names[v - 1] for v in y], name="class"),
            pd.Series(labels, name="cluster"))

### Looking at the clusters

In [ ]:
counts = np.bincount(labels, minlength=K)

fig, axes = plt.subplots(K, 11, figsize=(15, K * 1.45),
                         gridspec_kw={"width_ratios": [0.35] + [1] * 10})

for row in range(K):
    axes[row, 0].axis("off")
    axes[row, 0].text(0.5, 0.5, f"cluster {row}\n{counts[row]} imgs",
                      ha="center", va="center", fontsize=8)

    members = np.where(labels == row)[0][:10]

    for col in range(1, 11):
        axes[row, col].axis("off")
        if col - 1 < len(members):
            axes[row, col].imshow(X_img[members[col - 1]])

plt.tight_layout()
plt.show()

### Results

k-means, given no labels at all, recovers most of the ten classes.
But it is not perfect.
There are visually similar pairs.